# Assessment 1: Analysing historical data with system performance - Phase 2

**Student ID:** 35721588  
**Unit:** ITO5202  
**Teaching Period:** 5, 2026

**Dataset:** Brazilian E-Commerce Public Dataset by Olist  
**Source:** https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce

---

## Contents

**Part A: Analytical query design and implementation**
1. Business query design and justification
2. DataFrame API implementation
3. Spark SQL implementation
4. Result validation and API comparison

**Part B: System perspective and performance analysis**
1. Partitioning strategy analysis
2. Execution time benchmarking
3. Execution plan interpretation
4. DAG analysis via the Spark Web UI

---

## Environment and configuration

### Execution environment

For this project, we plan to run Spark in **local mode** on a single machine. In this setup, Spark does not create separate executor JVMs. Instead, the driver process handles the computation itself and uses the machine’s available logical CPU cores to run tasks in parallel. Because of this, the main memory setting that matters for our setup is spark.driver.memory, so we do not need to configure executor memory separately.

| Property | Value |
|---|---|
| Machine | MacBook Air (Retina, 13-inch, 2018) |
| Processor | 1.6 GHz dual-core Intel Core i5 |
| Logical cores | 4 |
| Physical memory | 8 GB |
| Operating system | macOS Sonoma 14.7.8 |
| Java | Eclipse Temurin JDK 17 (x64) |
| Python | 3.11.9 |
| PySpark | 3.5.1 |
| Spark master | `local[*]` |

The environment details are generated directly in the notebook rather than written in manually. This means the values referred to later in the Part B benchmarking discussion can be checked against the notebook output.

In [1]:
# Import packages
import os
import time
from statistics import median
import pandas as pd

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.types import *

# Set up data directory folder path
DATA_DIR = "data"


---

## SparkSession configuration

For this project, there are three key Spark settings which we changed from their default values due to how they affect the behaviour we want to examine later in Part B.

**`spark.driver.memory = 3g`.** Our machine has 8 GB of RAM, which also needs to support the computer's other processes. Allocating too much memory to Spark could slow  down our execution significantly. This matters to us beyond a speed perspective, since Part B.2 compares execution times. More specifically, this would make our results less useful because they could reflect memory pressure rather than Spark's actual processing behaviour.

**`spark.sql.shuffle.partitions = 4`.** This setting determines how many partitions Spark creates after a shuffle. The default is 200, which makes more sense for a much larger cluster than for the local environment we are using here. With four available task slots and a working set of roughly 110,000 rows after joining and filtering, using 200 partitions would create many very small tasks and add unnecessary scheduling overhead.

**`spark.sql.adaptive.enabled = false`.** Spark's Adaptive Query Execution (AQE) can change the physical execution plan while a query is running. On one hand, this can improve performance, but on the other hand, it makes the execution harder to compare with the plan shown by `explain(extended=True)`. Because Parts B.3 and B.4 require us to examine the physical plan and its corresponding DAG, AQE is turned off so that the printed plan and the executed plan remain consistent. This also means that the partition counts discussed in Part B.1 reflect the values we set orselves rather than values Spark changes during execution.


In [2]:
# Spark session build with local mode, 4 task slots, AQE off (as explained above)
spark = SparkSession.builder \
    .appName("ITO5202-A1-Olist-Freight") \
    .master("local[*]") \
    .config("spark.driver.memory", "3g") \
    .config("spark.sql.shuffle.partitions", 4) \
    .config("spark.sql.adaptive.enabled", False) \
    .config("spark.sql.session.timeZone", "UTC") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
spark

26/09/21 20:46:52 WARN Utils: Your hostname, Mounishas-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 192.168.0.137 instead (on interface en0)
26/09/21 20:46:52 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/21 20:46:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
sc = spark.sparkContext

# Print the main Spark environment settings
print("Spark version:", spark.version)
print("Master:", sc.master)
print("Available task slots:", sc.defaultParallelism)
print("Driver memory:", spark.conf.get("spark.driver.memory"))
print("Shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))
print("AQE enabled:", spark.conf.get("spark.sql.adaptive.enabled"))
print("Broadcast join threshold (bytes):", spark.conf.get("spark.sql.autoBroadcastJoinThreshold"))

# Web UI link
print("Web UI:", sc.uiWebUrl)

Spark version: 3.5.1
Master: local[*]
Available task slots: 4
Driver memory: 3g
Shuffle partitions: 4
AQE enabled: false
Broadcast join threshold (bytes): 10485760b
Web UI: http://192.168.0.137:4040


In [4]:
# Confirms the session can run a job end to end
spark.range(10).count()

10


---

## Data Loading

### Defining schemas

Our seven in-scope source files are loaded in using manually defined schemas instead of `inferSchema=True`. This is because by us using schema inference, Spark has to inspect the data before it is able to load it, thus adding extra work (this is particularly important for the geolocation file, which contains over one million rows).

Furthermore, if we were to allow Spark to infer the schema automatically, the same column in different data files could be interpreted as being of different data types, which can have downstream impacts on analysis when we try to do joins. Instead, by defining the schema ourselves, we are able to ensure consistency in data types. 

We also note that as per our proposal, we are looking to only use seven of the nine available data files, as do not wish to include the payments and reviews dataset as they fall outside the scope of our analysis. As such, we do not read in these datasets such as to avoid unneccessary extra work.

#### _Note_

Given a heading error in Section 3 of our original proposal, we need to make a correction in our dataset list. The first table in Section 3 of our approved proposal is labelled `olist_geolocation_dataset.csv`, but the columns listed underneath it actually belong to `olist_order_items_dataset.csv`. The geolocation dataset then appears again later in the sme section of our proposal with the correct columns.

We note that the approved proposal has been kept unchanged in `proposal/proposal.md` for consistency. However, the schemas defined below use the actual, correct columns from each source file, which were checked against the downloaded dataset.

#### _Note 2_

The source Kaggle file uses the column names `product_name_lenght` and `product_description_lenght`. However, in our approved proposal, we listed these names with the correct spelling of "length". 

Now, for the purposes of our analysis, the schema needs to match the actual CSV headers exactly, thus creating a mismatch with our earlier proposal. To keep the rest of the notebook easier to read, these two columns are renamed immediately after loading.

In [5]:
# Define the schemas manually so Spark does not have to infer them

order_items_schema = StructType([
    StructField("order_id", StringType(), False),
    StructField("order_item_id", IntegerType(), False),
    StructField("product_id", StringType(), True),
    StructField("seller_id", StringType(), True),
    StructField("shipping_limit_date", TimestampType(), True),
    StructField("price", DoubleType(), True),
    StructField("freight_value", DoubleType(), True),
])

orders_schema = StructType([
    StructField("order_id", StringType(), False),
    StructField("customer_id", StringType(), False),
    StructField("order_status", StringType(), True),
    StructField("order_purchase_timestamp", TimestampType(), True),
    StructField("order_approved_at", TimestampType(), True),
    StructField("order_delivered_carrier_date", TimestampType(), True),
    StructField("order_delivered_customer_date", TimestampType(), True),
    StructField("order_estimated_delivery_date", TimestampType(), True),
])

customers_schema = StructType([
    StructField("customer_id", StringType(), False),
    StructField("customer_unique_id", StringType(), True),
    StructField("customer_zip_code_prefix", IntegerType(), True),
    StructField("customer_city", StringType(), True),
    StructField("customer_state", StringType(), True),
])

sellers_schema = StructType([
    StructField("seller_id", StringType(), False),
    StructField("seller_zip_code_prefix", IntegerType(), True),
    StructField("seller_city", StringType(), True),
    StructField("seller_state", StringType(), True),
])

geolocation_schema = StructType([
    StructField("geolocation_zip_code_prefix", IntegerType(), True),
    StructField("geolocation_lat", DoubleType(), True),
    StructField("geolocation_lng", DoubleType(), True),
    StructField("geolocation_city", StringType(), True),
    StructField("geolocation_state", StringType(), True),
])

products_schema = StructType([
    StructField("product_id", StringType(), False),
    StructField("product_category_name", StringType(), True),
    StructField("product_name_lenght", IntegerType(), True),
    StructField("product_description_lenght", IntegerType(), True),
    StructField("product_photos_qty", IntegerType(), True),
    StructField("product_weight_g", IntegerType(), True),
    StructField("product_length_cm", IntegerType(), True),
    StructField("product_height_cm", IntegerType(), True),
    StructField("product_width_cm", IntegerType(), True),
])

category_schema = StructType([
    StructField("product_category_name", StringType(), True),
    StructField("product_category_name_english", StringType(), True),
])

In [6]:
# Helper function to read a CSV
def load_csv(filename, schema):
    return (spark.read
            .option("header", True)
            .schema(schema)
            .csv(f"{DATA_DIR}/{filename}"))

order_items = load_csv("olist_order_items_dataset.csv", order_items_schema)
orders      = load_csv("olist_orders_dataset.csv", orders_schema)
customers   = load_csv("olist_customers_dataset.csv", customers_schema)
sellers     = load_csv("olist_sellers_dataset.csv", sellers_schema)
geolocation = load_csv("olist_geolocation_dataset.csv", geolocation_schema)
categories  = load_csv("product_category_name_translation.csv", category_schema)

# Correct the misspelled column names present in the source file headers
products = (load_csv("olist_products_dataset.csv", products_schema)
            .withColumnRenamed("product_name_lenght", "product_name_length")
            .withColumnRenamed("product_description_lenght", "product_description_length"))

In [7]:
# Check row counts against the expected values
order_items_count = order_items.count()
orders_count = orders.count()
customers_count = customers.count()
sellers_count = sellers.count()
geolocation_count = geolocation.count()
products_count = products.count()
categories_count = categories.count()

rows = [
    {
        "dataset": "order_items",
        "expected": 112650,
        "actual": order_items_count,
        "match": order_items_count == 112650,
        "columns": len(order_items.columns)
    },
    {
        "dataset": "orders",
        "expected": 99441,
        "actual": orders_count,
        "match": orders_count == 99441,
        "columns": len(orders.columns)
    },
    {
        "dataset": "customers",
        "expected": 99441,
        "actual": customers_count,
        "match": customers_count == 99441,
        "columns": len(customers.columns)
    },
    {
        "dataset": "sellers",
        "expected": 3095,
        "actual": sellers_count,
        "match": sellers_count == 3095,
        "columns": len(sellers.columns)
    },
    {
        "dataset": "geolocation",
        "expected": 1000163,
        "actual": geolocation_count,
        "match": geolocation_count == 1000163,
        "columns": len(geolocation.columns)
    },
    {
        "dataset": "products",
        "expected": 32951,
        "actual": products_count,
        "match": products_count == 32951,
        "columns": len(products.columns)
    },
    {
        "dataset": "categories",
        "expected": 71,
        "actual": categories_count,
        "match": categories_count == 71,
        "columns": len(categories.columns)
    }
]

pd.DataFrame(rows)

,dataset,expected,actual,match,columns
0,order_items,112650,112650,True,7
1,orders,99441,99441,True,8
2,customers,99441,99441,True,5
3,sellers,3095,3095,True,4
4,geolocation,1000163,1000163,True,5
5,products,32951,32951,True,9
6,categories,71,71,True,2



---

## Data quality assessment

Our approved proposal identified a few data quality issues that we will need to check before we can safely begin our analysis. 

As such, it is important for us to now look at these issues in the loaded data, measure how much of the data is affected, and consider the cleaning and aggregation choices we will need to make later when building our main analysis dataset.


In [8]:
# Check the different order statuses
orders.groupBy("order_status") \
      .count() \
      .orderBy(F.desc("count")) \
      .show()

+------------+-----+
|order_status|count|
+------------+-----+
|   delivered|96478|
|     shipped| 1107|
|    canceled|  625|
| unavailable|  609|
|    invoiced|  314|
|  processing|  301|
|     created|    5|
|    approved|    2|
+------------+-----+



In [9]:
# Check missing delivery-related timestamps
orders.select(
    F.sum(F.col("order_approved_at").isNull().cast("int")).alias("missing_approved"),
    F.sum(F.col("order_delivered_carrier_date").isNull().cast("int")).alias("missing_carrier"),
    F.sum(F.col("order_delivered_customer_date").isNull().cast("int")).alias("missing_delivered"),
    F.sum(F.col("order_estimated_delivery_date").isNull().cast("int")).alias("missing_estimated")
).show()

+----------------+---------------+-----------------+-----------------+
|missing_approved|missing_carrier|missing_delivered|missing_estimated|
+----------------+---------------+-----------------+-----------------+
|             160|           1783|             2965|                0|
+----------------+---------------+-----------------+-----------------+



In [10]:
# Check how many geolocation rows there are for each postcode prefix
geo_stats = geolocation.groupBy("geolocation_zip_code_prefix").count()

total_geo_rows = geolocation.count()
distinct_postcodes = geo_stats.count()

print("Total geolocation rows:", total_geo_rows)
print("Distinct postcode prefixes:", distinct_postcodes)

geo_stats.agg(
    F.avg("count").alias("average rows per postcode"),
    F.max("count").alias("maximum rows per postcode")
).show()

Total geolocation rows: 1000163
Distinct postcode prefixes: 19015


+-------------------------+-------------------------+
|average rows per postcode|maximum rows per postcode|
+-------------------------+-------------------------+
|       52.598632658427555|                     1146|
+-------------------------+-------------------------+



In [11]:
# Number of different customer postcode prefixes
customer_postcodes = customers.select("customer_zip_code_prefix").distinct().count()

print("Distinct customer postcode prefixes:", customer_postcodes)


# Check how customers are distributed across states
customers.groupBy("customer_state") \
         .count() \
         .orderBy(F.desc("count")) \
         .show(10)

Distinct customer postcode prefixes: 14994
+--------------+-----+
|customer_state|count|
+--------------+-----+
|            SP|41746|
|            RJ|12852|
|            MG|11635|
|            RS| 5466|
|            PR| 5045|
|            SC| 3637|
|            BA| 3380|
|            DF| 2140|
|            ES| 2033|
|            GO| 2020|
+--------------+-----+
only showing top 10 rows



In [12]:
# Check for missing product information used later in the analysis

products.select(
    F.sum(F.col("product_category_name").isNull().cast("int")).alias("missing_category"),
    F.sum(F.col("product_weight_g").isNull().cast("int")).alias("missing_weight"),
    F.sum(F.col("product_length_cm").isNull().cast("int")).alias("missing_length"),
    F.sum(F.col("product_height_cm").isNull().cast("int")).alias("missing_height"),
    F.sum(F.col("product_width_cm").isNull().cast("int")).alias("missing_width")
).show()

+----------------+--------------+--------------+--------------+-------------+
|missing_category|missing_weight|missing_length|missing_height|missing_width|
+----------------+--------------+--------------+--------------+-------------+
|             610|             2|             2|             2|            2|
+----------------+--------------+--------------+--------------+-------------+




---

## Data quality findings

**Order completeness:** There are 99,441 orders in total, with 96,478 marked as `delivered` (97.0%). However, 2,965 orders have missing `order_delivered_customer_date`, which exceeds the number of non-delivered orders by eight. This means that in our data, there are eight orders which were marked as delivered even though there was no recorded delivery timestamp. Furthermore, there is no indication from the data as to what could be causing this (e.g. delivery error, system issue, , etc.).
Since we do not have a clear cause, we should treat this as a data integrity issue, and thus want to filter our analysis for both `oder_status = 'delivered'`, as well as a non-null delivery timestamp. 

**Geolocation duplication:** The geolocation dataset has 1,000,163 rows, however it only contains 19,015 unique postcode prefixes. That means that on average, each unique postcode appears ~52.6 times in the dataset, with the most common postcode appearing 1,146 times. Given this, if we were to join this dataset directly, it would result in a large number of duplicate matches.
Thus, we want to reduce our geolocation data to one row per postcode prefix before conducting any joins. As an additional benefit, at this smaller size, it is also small enough to be used as a broadcast lookup rather than requiring both sides of the join to be shuffled.

**Cardinality of partitioning column:** `customer_zip_code_prefix` contains 14,994 unique values, which we can see is very close to the ~15,000 expected values we mentioned in the proposal. Another thing we briefly mentioned in the proposal was the uneven distribution of customers across states. 
From our above data exploration, we can see a much clearer picture of the customer distribution, with São Paulo containing 41,746 customers, which accounts for ~42% of the total, while SP, RJ and MG together account for aother ~66.6%. 
This uneven distribution is important for us later when we consider the hash and range partitioning comparison in Part B.1, because range partitioning may produce less balanced partitions when the values themselves are unevenly distributed.

**Product attribute completeness:** We can see that there are 610 products (1.9%) with no category name, while two products are missing physical dimension values. This means measures that depend on package volume or weight cannot be calculated for those rows. 
As such, instead of filling in estimated values, we want to exclude these rows from the relevant weight-based calculations, with the number of affected records being reported with the results.

**Category translation coverage.** The translation table contains 71 category names, but the products table contains two categories that are missing from it: `pc_gamer` and `portateis_cozinha_e_preparadores_de_alimentos`. Products in these two categories only retain their Portuguese name and receive a null English translation.


---

## Part A.1: Business Query Design

### Analytical Question

When considering any analysis, it is important for us to first consider the business context of the data. 

We know that Olist operates as a marketplace intermediary; meaning that sellers are able to set their own prices vand Olist charges the customer a freight amount per item. However, what we aren't able to immediately ascertain is how to compare these values, any indication of freight profit, cost recovery or whether a shipment was priced correctly.That is to say, the same freight charge can represent something quite different for a light product and a heavy product, or for shipments travelling between different states.

Thus, for our analysis, freight is therefore compared relative to product weight and shipping lane.

From here, we seek to understand the answer to one key question through our query:

> **For each shipping lane (seller state to customer state) and each calendar quarter, what is the mean freight charged per kilogram, how does it compare to other lanes active in the same quarter, and how has it changed from the same lane's previous period?**

In order for us to answer this, our analysis first groups order items by shipping lane and quarter so that we have final output contains one row for each qualifying lane and quarter. We then want to calculate freight charged per kilogram, rank the qualifying lanes within each quarter, and compare each lane with its own result from the previous quarter.

#### **How we can interpret the results**

Our query can provide us with two different types of comparison:

- The **within-quarter** ranking, which compares shipping lanes with one another. We note that this needs to be interpreted carefully because freight charges are affected by distance and other shipment characteristics. For example, a shipment within one state is not directly comparable with one travelling across Brazil. The ranking is therefore used mainly to show the distribution of freight per kilogram and identify lanes at the higher or lower ends of that distribution.
- The **within-lane** comparison, which is useful for looking at change over time. Because the seller and customer states remain the same with lanes, geography is more consistent when a lane is compared with its own earlier result. A large change may therefore suggest a change in factors such as the products being shipped, package size, carrier arrangements or freight pricing.

Importantly, we need to keep in mind that a higher or lower value is not treated as automatically "good" or "bad". Rather, our main purpose should be the identification of patterns that may be worth investigating further. We should also note that distance_km is calculated from the postcode-prefix centroid values, so it utilises straight-line distance rather than the actual shipment distance (Which is unknown from our dataset). However, we have elected to retain it as supporting context even though it does not form part of our primary measure.

#### **What the analysis can tell us**

The result of our analysis can be leverage as a potential screening tool which can highlight lanes that:

- Have relatively high or low freight per kilogram within a quarter; or
- Show a noticeable change compared with their previous observed quarter.

The main purpose of the analysis is therefore to reduce a large number of individual order items into a smaller set of lane-quarter results that can be investigated more closely.

#### **Operations used**

From the above, it becomes clear that the results we need cannot be produced using a single aggregation. This is due to the fact that the required information is spread across several datasets and the analysis includes both group-level and time-based comparisons.

| Operation | Use in the analysis |
|---|---|
| **Joins** | `order_items` is combined with `orders` for order dates and status, `customers` and `sellers` for the shipping lane, and `products` for product weight. |
| **Derived columns** | Shipping lane, weight in kilograms, freight per kilogram and purchase quarter are calculated from the source columns. |
| **Aggregation** | Order items are grouped by shipping lane and quarter to produce the final lane-quarter level measures. |
| **Post-aggregation filtering** | Lane-quarter groups with too few items are removed after aggregations due to the fact that the minimum count applies to the group rather than an individual row. |
| **Window functions** | `RANK()` compares lanes within the same quarter, while `LAG()` retrieves the previous available result for each lane. |
| **Time-based analysis** | Calendar quarters are derived from `order_purchase_timestamp` and used to compare results over time. | 

We can see from the above that our query plans to use six of the operation types listed in the assessment requirements, which is above the required minimum.

#### **Why Spark is appropriate for the analysis**

It is obvious that our chosen dataset is small enough to run on one machine, and thus distributed computing is not neccessarily a requirement purely driven by dataset size. 

In saying that, Spark is still useful because the query contains several operations that are important in distributed data processing. Our analysis combines datasets with different sizes, performs joins on high-cardinality keys, aggregates the resulting records, and then applies window functions that require the data to be organised in different ways. 

Some of these operations can require Spark to redistribute data between partitions. The value of Spark here is therefore not that the dataset could not be processed without it, but that the query demonstrates the types of operations and execution decisions that become important with larger distributed datasets.

#### **Filtering decisions**

- **Delivered orders only:** Our data quality checks in the previous section of tis notebook showed that our dataset contained eight orders which were marked as `delivered` despite having no recorded customer delivery timestamp. Thus, our analysis requires both `order_status = 'delivered'` and a non-null `order_delivered_customer_date`. 
- **Recorded product weight:** Freight per kilogram cannot be calculated when `product_weight_g` is missing. As such, records where `product_weight_g` is missing are excluded from calculations that require weight rather than simply estimating a value for them. 
- **Minimum lane-quarter volume:** Lane-quarter groups containing only a small number of items can produce unstable or extreme results. Thus, a minimum item count is therefore applied after aggregation so the ranking is based on groups with a more meaningful amount of data.


---

## Part A.2: DataFrame API implementation

### Building the analytical base view

The information we need for our analysis is spread across several source files, so the first step for us is to combine them into a single row-level DataFrame.

This base view brings together the order, customer, seller and product information needed for our query, and also applies the filtering decisions we identified earlier.

The remaining derived columns, such as shipping lane, product weight in kilograms and purchase quarter, are also created at this stage so that we can leverage them later in our analysis.

The completed base view is cached because it is used repeatedly in later sections, including the DataFrame and SQL versions of the query, the equivalence check, partitioning experiments and execution-plan analysis. This avoids rebuilding the same joins and derived columns each time.

#### **Join approach**

Since our source tables are not all the same size, the joins need to be handled differently depending on the data involved. 

The larger joins, particularly those involving `order_items`, `orders` and `customers`, use high-cardinality keys and may require Spark to redistribute records between partitions. This makes them more expensive than joins involving the smaller lookup tables. 

The smaller datasets on the other hand, such as sellers, products and the reduced geolocation lookup, are small enough to be candidates for broadcast joins, which avoids shuffling both sides.

In particular, we had earlier discussed the pre-processing work required for the geolocation dataset and outlined the potential for duplicate matching due to multiple rows containing the same postcode prefix. In the context of our join approach, we can see that this additional processing step has the added benefit of making the lookup much smaller, and thus making it suitable for a broadcast join as well.


In [13]:
# Reduce the geolocation data to one row for each postcode prefix
geo_lookup = geolocation.groupBy("geolocation_zip_code_prefix").agg(
    F.avg("geolocation_lat").alias("lat"),
    F.avg("geolocation_lng").alias("lng")
)

# Cache the geolocation data
geo_lookup.cache()

geo_count = geo_lookup.count()
print("Geolocation lookup rows:", geo_count)

Geolocation lookup rows: 19015


In [14]:
# Keep orders that were delivered and have a delivery date
orders_delivered = orders.filter(
    (F.col("order_status") == "delivered") &
    (F.col("order_delivered_customer_date").isNotNull())
)

delivered_count = orders_delivered.count()
print("Delivered orders retained:", delivered_count)

Delivered orders retained: 96470


In [15]:
# Add the two category translations missing from the source file
missing_categories = spark.createDataFrame(
    [
        ("pc_gamer", "pc_gamer"),
        ("portateis_cozinha_e_preparadores_de_alimentos",
         "kitchen_portables_and_food_preparers")
    ],
    ["product_category_name", "product_category_name_english"]
)

categories_complete = categories.unionByName(missing_categories)

In [16]:
# Only keep the columns needed from each table for the joins
orders_join = orders_delivered.select(
    "order_id",
    "customer_id",
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
)

customers_join = customers.select(
    "customer_id",
    "customer_state",
    "customer_zip_code_prefix"
)

sellers_join = sellers.select(
    "seller_id",
    "seller_state",
    "seller_zip_code_prefix"
)

products_join = products.select(
    "product_id",
    "product_category_name",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm"
)

categories_join = categories_complete.select(
    "product_category_name",
    "product_category_name_english"
)

In [17]:
# Build the main row-level dataset
base = order_items.join(
    orders_join,
    on="order_id",
    how="inner"
)

base = base.join(
    customers_join,
    on="customer_id",
    how="inner"
)

# These tables are small, so request broadcast joins
base = base.join(
    F.broadcast(sellers_join),
    on="seller_id",
    how="inner"
)

base = base.join(
    F.broadcast(products_join),
    on="product_id",
    how="left"
)

base = base.join(
    F.broadcast(categories_join),
    on="product_category_name",
    how="left"
)

print("Base view rows:", base.count())

Base view rows: 110189


In [18]:
# Separate postcode lookups for customers and sellers
customer_geo = geo_lookup.select(
    F.col("geolocation_zip_code_prefix").alias("customer_zip_code_prefix"),
    F.col("lat").alias("cust_lat"),
    F.col("lng").alias("cust_lng")
)

seller_geo = geo_lookup.select(
    F.col("geolocation_zip_code_prefix").alias("seller_zip_code_prefix"),
    F.col("lat").alias("sell_lat"),
    F.col("lng").alias("sell_lng")
)

In [19]:
# Add approximate coordinates for both ends of the shipping lane
base = base.join(
    F.broadcast(customer_geo),
    on="customer_zip_code_prefix",
    how="left"
)

base = base.join(
    F.broadcast(seller_geo),
    on="seller_zip_code_prefix",
    how="left"
)

In [20]:
# Create the columns needed for analysis
base_view = base.withColumn(
    "lane",
    F.concat_ws(" -> ", F.col("seller_state"), F.col("customer_state"))
)

base_view = base_view.withColumn(
    "purchase_quarter",
    F.concat_ws(
        "-",
        F.year("order_purchase_timestamp"),
        F.concat(F.lit("Q"), F.quarter("order_purchase_timestamp"))
    )
)

base_view = base_view.withColumn(
    "weight_kg",
    F.col("product_weight_g") / 1000.0
)

base_view = base_view.withColumn(
    "freight_per_kg",
    F.when(
        F.col("weight_kg") > 0,
        F.col("freight_value") / F.col("weight_kg")
    )
)

base_view = base_view.withColumn(
    "freight_to_price_ratio",
    F.when(
        F.col("price") > 0,
        F.col("freight_value") / F.col("price")
    )
)

base_view = base_view.withColumn(
    "package_volume_cm3",
    F.col("product_length_cm") *
    F.col("product_height_cm") *
    F.col("product_width_cm")
)

In [21]:
# Approximate straight-line distance between the seller and customer
base_view = base_view.withColumn(
    "distance_km",
    F.lit(6371.0) * 2 * F.asin(
        F.sqrt(
            F.pow(
                F.sin(
                    F.radians(F.col("cust_lat") - F.col("sell_lat")) / 2
                ),
                2
            )
            +
            F.cos(F.radians(F.col("sell_lat"))) *
            F.cos(F.radians(F.col("cust_lat"))) *
            F.pow(
                F.sin(
                    F.radians(F.col("cust_lng") - F.col("sell_lng")) / 2
                ),
                2
            )
        )
    )
)

base_view = base_view.withColumn(
    "is_intrastate",
    F.col("seller_state") == F.col("customer_state")
)

In [22]:
# Keep the columns that we will later
base_view = base_view.select(
    "order_id",
    "order_item_id",
    "lane",
    "seller_state",
    "customer_state",
    "customer_zip_code_prefix",
    "seller_zip_code_prefix",
    "purchase_quarter",
    "order_purchase_timestamp",
    "price",
    "freight_value",
    "weight_kg",
    "freight_per_kg",
    "freight_to_price_ratio",
    "package_volume_cm3",
    "distance_km",
    "is_intrastate",
    "product_category_name",
    "product_category_name_english"
)


#### **Caching decision**

Here, `base_view` is cached because given that we need to reuse it throughout the rest of our analysis, and it would take several steps to construct each time.

Since Spark transformations are not actually executed until an action is called, without caching, repeated actions on `base_view` would result in Spark having to work through the same upstream processing multiple times, including reading the source data, reducing the geolocation dataset, performing the joins and calculating the derived columns.

Also, caching is reasonable here because the final base view is much smaller than the raw source data. It contains ~110,000 rows, and has already been reduced to only the columns we likely need later in the notebook. By comparison, if we were to cache the original source datasets, this would provide considerably less benefit because most of the columns there are only used once when building the base view.

Furthermore, `geo_lookup` is also cached because the same postcode lookup is used twice: once for the customer location and once for the seller location. This avoids repeating the aggregation of the original geolocation dataset for both joins.


In [23]:
# Confirm the joins preserved one row per order item before caching
print("Base view rows:", base_view.count())
print("Distinct order item keys:", 
      base_view.select("order_id", "order_item_id").distinct().count())

Base view rows: 110189


Distinct order item keys: 110189


In [24]:
# Cache the finished base view
base_view.cache()

# Run count() so the cache is actually populated
base_count = base_view.count()

print("Base view rows:", base_count)
print("Number of columns:", len(base_view.columns))

Base view rows: 110189
Number of columns: 19


In [25]:
# Main analysis columns
base_view.select(
    "lane",
    "purchase_quarter",
    "price",
    "freight_value",
    "weight_kg",
    "freight_per_kg",
    "distance_km"
).show(10, truncate=False)



+--------+----------------+------+-------------+---------+------------------+------------------+
|lane    |purchase_quarter|price |freight_value|weight_kg|freight_per_kg    |distance_km       |
+--------+----------------+------+-------------+---------+------------------+------------------+
|SP -> RJ|2017-Q3         |58.9  |13.29        |0.65     |20.446153846153845|301.50468067509604|
|SP -> SP|2017-Q2         |239.9 |19.93        |30.0     |0.6643333333333333|585.5639365089672 |
|MG -> MG|2018-Q1         |199.0 |17.87        |3.05     |5.859016393442624 |312.34351136818617|
|SP -> SP|2018-Q3         |12.99 |12.79        |0.2      |63.949999999999996|293.16841989111373|
|PR -> SP|2017-Q1         |199.9 |18.14        |3.75     |4.8373333333333335|646.1634625711988 |
|SP -> MG|2017-Q2         |21.9  |12.69        |0.45     |28.2              |161.85424084700483|
|SP -> SP|2017-Q4         |19.9  |11.85        |0.2      |59.24999999999999 |484.860164492648  |
|SP -> SP|2018-Q3         |810

In [26]:
# Check how many rows are missing values
base_view.select(
    F.count("*").alias("total_rows"),
    F.sum(
        F.col("freight_per_kg").isNull().cast("int")
    ).alias("missing_freight_per_kg"),
    F.sum(
        F.col("distance_km").isNull().cast("int")
    ).alias("missing_distance"),
    F.sum(
        F.col("product_category_name").isNull().cast("int")
    ).alias("missing_category"),
    F.sum(
    F.col("product_category_name_english").isNull().cast("int")
).alias("missing_english_category")
).show()

+----------+----------------------+----------------+----------------+------------------------+
|total_rows|missing_freight_per_kg|missing_distance|missing_category|missing_english_category|
+----------+----------------------+----------------+----------------+------------------------+
|    110189|                    26|             536|            1537|                    1537|
+----------+----------------------+----------------+----------------+------------------------+



#### **Defining the freight per kilogram measure**

Before aggregating the data, we still need to define what our "freight per kilogram" measure is actually measuring so that our resulting analysis and interpretation is an accurate reflection of the data itself. Fundamentally, there are two reasonable ways this cmeasure could be defined, and given that they can produce quite different results, it is important for us to carefully step through them before making a decision.

The first option is to average the `freight_per_kg` value calculated for each individual item. The key issue here is that very light items can produce extremely high ratios. For example, in the sample above, a 0.2 kg item has freight of 63.95 per kg, while a 30 kg item has a value of only 0.66 per kg. If these item-level ratios are averaged, both items have the same influence on the result even though their weights are very different. 

The second option is to divide the **total freight charged by the total weight shipped** within each lane-quarter. This gives an overall freight charge per kilogram across everything shipped in that group.

It is clear from the above that the second definition provides a much more robust analysis. However, we do need to note that by using the total freight divided by total weight definition, it also impacts how missing values should be handled. More specifically, rows with a missing or zero product weight must be removed before aggregation. Otherwise, their freight value would still contribute to total freight while adding no weight to the denominator, which would overstate freight per kilogram for that group.

We know already from our earlier data quality check that there are 26 rows in the base view where a valid freight-per-kilogram value could not be calculated, so these rows need to be excluded before the lane-quarter aggregation.

NOTE: Even though it is not used in this instance as a "freight per kilogram" definiiton, we still want to retain the average of the individual item-level ratios as a secondary measure due to the fact that comparing the two can help show where differences in item weights are having a strong effect on the result. 
We should be careful however, to clearly distinguish between the two measures so as to not conflate the two measure so as to not lead to incorrect conclusions.

In [27]:
# Keep rows where weight can be used in the calculation
base_weighted = base_view.filter(
    F.col("weight_kg") > 0
)

weighted_count = base_weighted.count()
base_count = base_view.count()

print("Rows retained for weight-based measures:", weighted_count)
print("Rows excluded:", base_count - weighted_count)

Rows retained for weight-based measures: 110163
Rows excluded: 26


#### **Establishing a minimum lane-quarter volume**

Now that we have confirmed the number of rows being retained, it is important to consider  reasonable threshold for the number of groups we are retaining. Rather than choosing a threshold arbitrarily, we first want to examine how the data is actually distributed across quarters and across lane-quarter combinations.

In [28]:
# Check how many rows are in each quarter

base_weighted.groupBy(
    "purchase_quarter"
).count().orderBy(
    "purchase_quarter"
).show()

+----------------+-----+
|purchase_quarter|count|
+----------------+-----+
|         2016-Q3|    3|
|         2016-Q4|  314|
|         2017-Q1| 5664|
|         2017-Q2|10051|
|         2017-Q3|13946|
|         2017-Q4|19875|
|         2018-Q1|23572|
|         2018-Q2|22644|
|         2018-Q3|14094|
+----------------+-----+



In [29]:
# Count how many items are in each lane and quarter
lane_quarter_sizes = base_weighted.groupBy(
    "lane",
    "purchase_quarter"
).count()

print("Total lane-quarter groups:", lane_quarter_sizes.count())

Total lane-quarter groups: 1856


In [30]:
# Look at the range of group sizes before choosing a minimum

lane_quarter_sizes.select(
    F.min("count").alias("minimum"),
    F.expr("percentile_approx(count, 0.25)").alias("25th percentile"),
    F.expr("percentile_approx(count, 0.50)").alias("median"),
    F.expr("percentile_approx(count, 0.75)").alias("75th percentile"),
    F.max("count").alias("maximum")
).show()

+-------+---------------+------+---------------+-------+
|minimum|25th percentile|median|75th percentile|maximum|
+-------+---------------+------+---------------+-------+
|      1|              2|     4|             18|   8053|
+-------+---------------+------+---------------+-------+



In [31]:
# See how many lane-quarter groups would remain at different minimum sizes

groups_10 = lane_quarter_sizes.filter(F.col("count") >= 10).count()
groups_20 = lane_quarter_sizes.filter(F.col("count") >= 20).count()
groups_30 = lane_quarter_sizes.filter(F.col("count") >= 30).count()
groups_50 = lane_quarter_sizes.filter(F.col("count") >= 50).count()
groups_100 = lane_quarter_sizes.filter(F.col("count") >= 100).count()

print("Groups with at least 10 items:", groups_10)
print("Groups with at least 20 items:", groups_20)
print("Groups with at least 30 items:", groups_30)
print("Groups with at least 50 items:", groups_50)
print("Groups with at least 100 items:", groups_100)

Groups with at least 10 items: 632
Groups with at least 20 items: 445
Groups with at least 30 items: 348
Groups with at least 50 items: 260
Groups with at least 100 items: 159


#### Choosing the minimum lane-quarter volume

We can see from the above checks that the lane-quarter group sizes are quite uneven. Across the 1,856 combinations, the median group contains only 4 items the 25th percentile contains 2, but the maximum number of items is 8,053. This means that many lane-quarter combinations are based on only a small number of shipments.

This is particularly important for our analysis because very small groups can produce very high or very low freight-per-kilogram values based on only a few items. Since we are ranking the lane-quarter results, these small groups could appear near the extremes of the ranking and can heavily skew our results.

We can also see the number of groups that are retained at various thresholds in the checks above. If we convert these figures to being percentages of the total, we have:

| Minimum items | Groups retained | Share of total |
|---|---:|---:|
| 10 | 632 | 34.1% |
| 20 | 445 | 24.0% |
| 30 | 348 | 18.8% |
| 50 | 260 | 14.0% |
| 100 | 159 | 8.6% |

For this analysis, we want to use a minimum of **30 items per lane-quarter**. This is above the 75th percentile group size, so it removes most of the very small combinations while still retaining 348 groups for comparison.

We believe that this is a practical choice to reduce the influence of groups based on only a handful of shipments while keeping enough lane-quarter combinations for the ranking and time-based analysis.

Importantly, this threshold also helps with the `LAG()` comparison. `LAG()` returns the previous **observed** result for each lane, which may not always be the immediately preceding calendar quarter. Requiring at least 30 items helps us to retain more active lanes, although gaps between observed quarters can still occur.

Furthermore, we can also see from our above checks that the earliest periods contain much less data than the later quarters. `2016-Q3` contains only 3 order items and `2016-Q4` contains 314, so it is likely that very few or no lane-quarter groups from this period will meet the 30-item threshold. As a result, most of our final analysis will be based on activity during 2017 and 2018.


In [32]:
MIN_ITEMS = 30

# Summarise the data by shipping lane and quarter
lane_quarter = base_weighted.groupBy(
    "lane",
    "seller_state",
    "customer_state",
    "purchase_quarter"
).agg(
    F.count("*").alias("item_count"),
    F.countDistinct("order_id").alias("order_count"),
    F.sum("freight_value").alias("total_freight"),
    F.sum("weight_kg").alias("total_weight_kg"),
    F.avg("freight_per_kg").alias("mean_item_freight_per_kg"),
    F.avg("freight_to_price_ratio").alias("mean_freight_to_price"),
    F.avg("distance_km").alias("mean_distance_km")
)

# Main freight per kg measure: total freight divided by total weight
lane_quarter = lane_quarter.withColumn(
    "freight_per_kg",
    F.col("total_freight") / F.col("total_weight_kg")
)

# Remove lane-quarter groups that are too small
lane_quarter = lane_quarter.filter(
    F.col("item_count") >= MIN_ITEMS
)

print("Qualifying lane-quarters:", lane_quarter.count())

Qualifying lane-quarters: 348


#### **Checking for a platform-wide trend**

Before we can compare each lane with its own previous quarter, we need to first know whether freight per kilogram was stable across the platform as a whole over the period.

In [33]:
# Look at the overall freight per kg and average item weight for each quarter
quarter_summary = base_weighted.groupBy(
    "purchase_quarter"
).agg(
    F.count("*").alias("items"),
    (F.sum("freight_value") / F.sum("weight_kg")).alias("quarter_baseline_fpk"),
    F.avg("weight_kg").alias("mean_weight_kg")
)

quarter_summary.select(
    "purchase_quarter",
    "items",
    F.round("quarter_baseline_fpk", 2).alias("overall_freight_per_kg"),
    F.round("mean_weight_kg", 2).alias("mean_weight_kg")
).orderBy(
    "purchase_quarter"
).show()

+----------------+-----+----------------------+--------------+
|purchase_quarter|items|overall_freight_per_kg|mean_weight_kg|
+----------------+-----+----------------------+--------------+
|         2016-Q3|    3|                  2.83|           1.0|
|         2016-Q4|  314|                  9.01|          2.18|
|         2017-Q1| 5664|                  8.31|          2.29|
|         2017-Q2|10051|                  8.54|          2.28|
|         2017-Q3|13946|                  8.62|          2.24|
|         2017-Q4|19875|                   9.3|          2.09|
|         2018-Q1|23572|                  9.39|          2.08|
|         2018-Q2|22644|                  9.92|          2.08|
|         2018-Q3|14094|                  12.3|          1.77|
+----------------+-----+----------------------+--------------+



#### **Adding a relative measure**

We can see from the above that the platform-wide freight-per-kilogram figure increased from 8.31 in 2017-Q1 to 12.30 in 2018-Q3, which is an increase of roughly 48%. Additionally, over the same period, the mean item weight fell from 2.29 kg to 1.77 kg, signifying a decrease of about 23%. 

We should note that hese movements are very likely to be related. More specifically, while freight charges do not necessarily decrease in direct proportion to parcel weight, some parts of a freight charge may remain similar regardless of how heavy an item is. Thus, as the average item weight falls, freight per kilogram can therefore increase even without an equivalent change in the underlying freight charge. 

In the context of our analysis, this likely means that the raw change for an individual lane should not be interpreted without considering what was happening across the platform at the same time. That is to say, if freight per kilogram are increasing across the board, part of a lane's increase could simply be attributed to that broader movement. 

In order for us to account for this, we can add two relative measures alongside the raw change:
- `relative_to_quarter`: which shows a lane's freight per kilogram as a multiple of the platform-wide freight-per-kilogram figure for the same quarter. 
- `relative_change`: which measures how that relative position changes between the lane's consecutive observed results. 

Through the above, we are able to then compare each lane against the platform figure from the same quarter in order to control for the overall shift in freight per kilogram over time, and ultimately provide a mechanism for us to distinguish a broad platform-wide movement from a change that is more specific to an individual lane.

In [34]:
# Add the platform-wide freight per kg for each quarter
lane_quarter = lane_quarter.join(
    F.broadcast(
        quarter_summary.select(
            "purchase_quarter",
            "quarter_baseline_fpk"
        )
    ),
    on="purchase_quarter",
    how="left"
)

lane_quarter = lane_quarter.withColumn(
    "relative_to_quarter",
    F.col("freight_per_kg") / F.col("quarter_baseline_fpk")
)

In [35]:
# Rank lanes within the same quarter
quarter_window = Window.partitionBy(
    "purchase_quarter"
).orderBy(
    F.desc("freight_per_kg")
)

# Used to count how many qualifying lanes are in each quarter
quarter_totals = Window.partitionBy(
    "purchase_quarter"
)

# Compare each lane with its previous observed result
lane_window = Window.partitionBy(
    "lane"
).orderBy(
    "purchase_quarter"
)

In [36]:
lane_quarter_ranked = lane_quarter.withColumn(
    "rank_in_quarter",
    F.rank().over(quarter_window)
)

lane_quarter_ranked = lane_quarter_ranked.withColumn(
    "lanes_in_quarter",
    F.count("*").over(quarter_totals)
)

lane_quarter_ranked = lane_quarter_ranked.withColumn(
    "prev_quarter",
    F.lag("purchase_quarter").over(lane_window)
)

lane_quarter_ranked = lane_quarter_ranked.withColumn(
    "prev_freight_per_kg",
    F.lag("freight_per_kg").over(lane_window)
)

lane_quarter_ranked = lane_quarter_ranked.withColumn(
    "prev_relative",
    F.lag("relative_to_quarter").over(lane_window)
)

lane_quarter_ranked = lane_quarter_ranked.withColumn(
    "change_pct",
    F.round(
        100 * (
            F.col("freight_per_kg") - F.col("prev_freight_per_kg")
        ) / F.col("prev_freight_per_kg"),
        1
    )
)

lane_quarter_ranked = lane_quarter_ranked.withColumn(
    "relative_change",
    F.col("relative_to_quarter") - F.col("prev_relative")
)

lane_quarter_ranked.cache()

print("Qualifying lane-quarters:", lane_quarter_ranked.count())

Qualifying lane-quarters: 348


In [37]:
# Look at the highest freight per kg lanes in the most recent full quarter
q2_2018 = lane_quarter_ranked.filter(
    F.col("purchase_quarter") == "2018-Q2"
)

print("=== 2018-Q2: lanes ranked by freight charged per kilogram ===")

q2_2018.select(
    "rank_in_quarter",
    "lane",
    "item_count",
    F.round("freight_per_kg", 2).alias("freight_per_kg"),
    F.round("mean_item_freight_per_kg", 2).alias("mean_item_ratio"),
    F.round("relative_to_quarter", 2).alias("relative_to_quarter"),
    F.round("mean_distance_km", 0).alias("avg_dist_km")
).orderBy(
    "rank_in_quarter"
).show(15, truncate=False)

=== 2018-Q2: lanes ranked by freight charged per kilogram ===
+---------------+--------+----------+--------------+---------------+-------------------+-----------+
|rank_in_quarter|lane    |item_count|freight_per_kg|mean_item_ratio|relative_to_quarter|avg_dist_km|
+---------------+--------+----------+--------------+---------------+-------------------+-----------+
|1              |MA -> SP|53        |61.07         |67.54          |6.16               |2329.0     |
|2              |PE -> MG|30        |49.21         |55.39          |4.96               |1700.0     |
|3              |PE -> SP|35        |41.06         |50.92          |4.14               |2118.0     |
|4              |RJ -> PE|30        |36.17         |62.69          |3.65               |1849.0     |
|5              |SP -> RN|70        |33.34         |116.88         |3.36               |2271.0     |
|6              |SP -> MA|85        |24.4          |78.59          |2.46               |2208.0     |
|7              |RJ -> RS|45 

In [38]:
# Largest changes from each lane's previous observed quarter
previous_results = lane_quarter_ranked.filter(
    F.col("prev_freight_per_kg").isNotNull()
)

print("=== Largest raw changes from previous observed quarter ===")

previous_results.select(
    "lane",
    "prev_quarter",
    "purchase_quarter",
    "item_count",
    F.round("prev_freight_per_kg", 2).alias("prev_fpk"),
    F.round("freight_per_kg", 2).alias("fpk"),
    "change_pct",
    "rank_in_quarter",
    "lanes_in_quarter"
).orderBy(
    F.desc(F.abs(F.col("change_pct")))
).show(15, truncate=False)

=== Largest raw changes from previous observed quarter ===
+--------+------------+----------------+----------+--------+-----+----------+---------------+----------------+
|lane    |prev_quarter|purchase_quarter|item_count|prev_fpk|fpk  |change_pct|rank_in_quarter|lanes_in_quarter|
+--------+------------+----------------+----------+--------+-----+----------+---------------+----------------+
|SP -> TO|2017-Q2     |2017-Q4         |32        |10.18   |32.81|222.4     |1              |58              |
|RJ -> PR|2018-Q2     |2018-Q3         |38        |15.18   |33.31|119.4     |2              |46              |
|MG -> PR|2017-Q1     |2017-Q2         |50        |7.87    |15.55|97.6      |7              |44              |
|RS -> RJ|2018-Q1     |2018-Q2         |50        |10.14   |19.51|92.4      |10             |60              |
|SP -> RN|2018-Q1     |2018-Q2         |70        |17.52   |33.34|90.2      |5              |60              |
|RS -> MG|2018-Q1     |2018-Q2         |40        |10

In [39]:
# Compare changes after accounting for the platform baseline in each quarter
relative_results = lane_quarter_ranked.filter(
    F.col("prev_relative").isNotNull()
)

print("=== Largest movements relative to the quarterly platform baseline ===")

relative_results.select(
    "lane",
    "prev_quarter",
    "purchase_quarter",
    "item_count",
    F.round("prev_relative", 2).alias("prev_relative"),
    F.round("relative_to_quarter", 2).alias("relative_to_quarter"),
    F.round("relative_change", 2).alias("relative_change"),
    "change_pct",
    "rank_in_quarter",
    "lanes_in_quarter"
).orderBy(
    F.desc(F.abs(F.col("relative_change")))
).show(15, truncate=False)

=== Largest movements relative to the quarterly platform baseline ===
+--------+------------+----------------+----------+-------------+-------------------+---------------+----------+---------------+----------------+
|lane    |prev_quarter|purchase_quarter|item_count|prev_relative|relative_to_quarter|relative_change|change_pct|rank_in_quarter|lanes_in_quarter|
+--------+------------+----------------+----------+-------------+-------------------+---------------+----------+---------------+----------------+
|SP -> TO|2017-Q2     |2017-Q4         |32        |1.19         |3.53               |2.34           |222.4     |1              |58              |
|SP -> TO|2017-Q4     |2018-Q1         |48        |3.53         |1.41               |-2.12          |-59.7     |19             |59              |
|SP -> RN|2018-Q1     |2018-Q2         |70        |1.87         |3.36               |1.5            |90.2      |5              |60              |
|SP -> MA|2017-Q2     |2017-Q3         |98        |2.9

#### **Reading the results**

Based on our analysis above, there are four key findings that we're able to identify:

**1. The within-quarter ranking is strongly related to distance, as expected.** We can see that in 2018-Q2, the eleven highest-ranked lanes all have mean distances between 1,100 km - 2,300 km, while RJ to MG, which is at rank 12 has a mean distance of just 385km. This supports our previous discussion around how a high ranking does not, by itself, mean that a lane is priced unusually.

**2. The two freight-per-kilogram measures can produce quite different results.** In the above analysis, for the SP to RN lane in 2018-Q2, the freight per kilogram is 33.34, while the mean of the individual item ratios is 116.88. This shows how very light items can have a large effect on the average of the individual ratios.

**3. The raw changes also need to be read against the broader platform trend.** Across the 280 comparisons with an observed result for previous periods, 170 are increases and 110 are decreases, with a median change of +5.0%. We note that freight per kilogram was also increasing across the platform over this period, so a positive raw change does not necessarily mean that a lane moved differently from the overall pattern.

**4. The relative measure gives additional context to these changes.** 
`relative_to_quarter` compares each lane with the platform-wide freight-per-kilogram figure from within the same quarter, while `relative_change` shows how that relative position changes over time. This helps us to account for the broader movement in the platform baseline. 

*Overall, we can see that the outputs above have helped us to narrow the analysis from 110,163 individual order items to a much smaller set of lane-quarter results worth examining more closely.* These include lanes that sit well above or below the platform baseline for a particular period and lanes whose relative position changes noticeably between observed quarters.


---

### How Spark executes and optimises our query

We can see from the above that our analysis moves through several key stages:
- The source files are joined to create `base_view`,
- The data is filtered and aggregated to lane-quarter level, and 
- Window functions are used for the ranking and previous-period comparisons.

An important feature of Spark is that it does not simply execute each line in the order it appears in the notebook. Instead, it leverages Catalyst to first build a logical plan, optimising it before producing a physical plan, which is then finally actually executed. Importantly, the optimisations below can therefore be linked directly to the different stages of our analysis, with the resulting physical plan being something we examine closely in Part B.3 later.

#### **Building the base view**

If we now turn to stepping through our analysis one step at a time, the first stage of our analysis involved combining information from the order items, orders, customers, sellers, products, category translation and geolocation datasets.

Before these joins could be carried out, we used a `select()` statement to keep only the columns that were needed later. For example, only five of the eight order columns and six of the product columns are taken into the joins. 

This is consistent with the *projection pruning* feature of Spark, where Catalyst works backwards from the required output to determine which columns are actually needed and avoids carrying unnecessary columns further through the query. Our manual `select()` statements make this explicit in the notebook, although Catalyst can also remove unused columns when it optimises the plan.

Our delivered-order filter also reduces the data before the main joins for a similar reason. We used the `order_status = 'delivered'` filter together with the non-null delivery timestamp removes incomplete orders before they enter the rest of the pipeline.

We can see that Catalyst operates in a somewhat similar way through *predicate pushdown*, where it attempts to move filters as close to the source as possible without changing the result, but still  reducing the rows passed into later joins.

#### **How the joins are handled**

The next stage to look at is the join stage. While there are several join algorithms available in Spark, the two most relevant to our query are broadcast joins and shuffle-based joins.

For smaller tables, such as the sellers, products, the category translation, and the reduced geolocation lookup tables, they can be explicitly wrapped in a broadcast join through `F.broadcast()`. Broadcasting sends a copy of the smaller table to the tasks working on the larger table, which avoids us having to redistribute both sides of the join by key.

On the other hand, larger joins such as the `order_items`, `orders` and `customers` ones are different. Since these tables contain more rows and use high-cardinality identifiers such as `order_id`, they are more likely to require Spark to use a shuffle-based strategy that redistributes rows by the join key.

Our geolocation lookup in particular is a useful example of preprocessing affecting the join strategy. The original geolocation file contains 1,000,163 rows because postcode prefixes appear repeatedly. We reduce this to 19,015 postcode-level records before joining. This step is necessary first to avoid duplicate matches, but it also makes the lookup much smaller. The reduced version can then be broadcast separately for the customer and seller postcode joins instead of joining the full geolocation table twice.

#### **Aggregation and filtering**

After the base view is created, we were then able to group the data by shipping lane and quarter. 

For aggregates such as total freight, total weight and item count, Spark is able to calculate partial results within each partition before combining them after a shuffle. That is to say that the majority of the aggregation work can be completed locally before data is moved between partitions, thereby reducing the amount of information that needs to be redistributed.

On the other hand, `countDistinct("order_id")` requires us to do some additional work because Spark needs to make sure an order is not counted more than once across partitions. It is retained because order count provides useful context alongside item count. 

The minimum volume rule (`item_count >= 30` ) is only applied after aggregationdue to the fact that `item_count` does not exist until each lane-quarter group has been formed. It is therefore equivalent to a SQL `HAVING` condition rather than a row-level filter.

#### **Adding the quarterly baseline**

Our next step was to calculate the platform-wide quarterly baseline `base_weighted`, using the total freight divided by total weight for each quarter.

In our above analysis, we saw that this gave us a small quarter-level table, which we could then join back to the qualifying lane-quarter results. Because the baseline containrf only one row per quarter, it could be explicitly broadcast when joined.

Importantly, keeping the baseline calculation separate is useful to us for two reasons:

1. It ensures the platform figure is calculated from all valid weighted rows rather than only from lane-quarter groups that pass the 30-item threshold. 
2. The same quarterly values used in our descriptive output are also used to calculate `relative_to_quarter`, so the definition remains consistent throughout the analysis.

#### **Applying the window functions**

Once we reduce our data to just the qualifying lane-quarter rows, we can calculate the remaining comparisons using window functions.

In order for us to do thus, there are two different types of groupings which are required:

- `purchase_quarter`: this is used to rank lanes against other lanes active in the same quarter and to count the number of qualifying lanes in that period.
- `lane`: which is used to retrieve each lane's previous observed result using `LAG()`.

These operations require Spark to organise the data according to the relevant window key, and the ordered windows also require rows to be sorted within those groups. Because the ranking and temporal comparison use different partitioning keys, Spark may need to redistribute the aggregated data again after the earlier `groupBy`.

#### **Effect of caching**

As part of our analysis, we made the decision to cache the `base_view` table given that we needed to reuse it in several later sections of the notebook. A key feature of Spark is that once the cache has been set, Spark can simply read the prepared analytical data from memory rather than repeating the upstream joins, geolocation aggregation and derived-column calculations. 

We note that this is likely to become particularly important for the DataFrame and SQL comparison and for the timed runs in Part B.2, where both implementations should begin from the same prepared input. 

Because `base_view` is cached, the execution plan for later queries starts from the cached DataFrame rather than showing all of the joins used to create it. This means the downstream plan mainly shows the aggregation and window operations. The join behaviour used to build `base_view` therefore needs to be examined separately when interpreting the physical plan in Part B.3.


---

## Part A.3: Spark SQL implementation

We note that in our earlier sections of this notebook, we have already built the analysis using the DataFrame API. Now, we want to recreate the same result using Spark SQL so that the two approaches can be compared directly in Part A.4.

In order for us to use SQL, the DataFrames we require need to be registered as temporary views first. In doing so, they are given a table name that can be referenced within SQL queries without creating or saving a separate dataset.

Both versions of our analysis use the same cached `base_view` that we defined earlier, which means they are able to both start from the same prepared data.

Here, the SQL query is split into several Common Table Expressions (CTEs) using `WITH`. Then, each CTE is able to handle one stage of the analysis, following the same general order as our previous DataFrame version:

- filter to rows with valid weight
- aggregate to lane-quarter level
- apply the 30-item minimum
- add the quarterly platform baseline
- calculate the ranking and previous-period comparisons

Keeping the SQL in these steps makes it easier for us to follow and also makes the logic easier to compare with our DataFrame implementation.


In [40]:
# Make base_view available to Spark SQL
base_view.createOrReplaceTempView("base_view")

spark.sql("SHOW TABLES").show()

+---------+---------+-----------+
|namespace|tableName|isTemporary|
+---------+---------+-----------+
|         |base_view|       true|
+---------+---------+-----------+



In [41]:
sql_query = """
WITH weighted_items AS (
    SELECT *
    FROM base_view
    WHERE weight_kg > 0
),

quarter_baseline AS (
    SELECT
        purchase_quarter,
        SUM(freight_value) / SUM(weight_kg) AS quarter_baseline_fpk
    FROM weighted_items
    GROUP BY purchase_quarter
),

lane_quarter AS (
    SELECT
        lane,
        seller_state,
        customer_state,
        purchase_quarter,
        COUNT(*) AS item_count,
        COUNT(DISTINCT order_id) AS order_count,
        SUM(freight_value) AS total_freight,
        SUM(weight_kg) AS total_weight_kg,
        SUM(freight_value) / SUM(weight_kg) AS freight_per_kg,
        AVG(freight_per_kg) AS mean_item_freight_per_kg,
        AVG(freight_to_price_ratio) AS mean_freight_to_price,
        AVG(distance_km) AS mean_distance_km
    FROM weighted_items
    GROUP BY
        lane,
        seller_state,
        customer_state,
        purchase_quarter
    HAVING COUNT(*) >= 30
),

with_baseline AS (
    SELECT
        lq.*,
        qb.quarter_baseline_fpk,
        lq.freight_per_kg / qb.quarter_baseline_fpk AS relative_to_quarter
    FROM lane_quarter lq
    LEFT JOIN quarter_baseline qb
        ON lq.purchase_quarter = qb.purchase_quarter
)

SELECT
    lane,
    seller_state,
    customer_state,
    purchase_quarter,
    item_count,
    order_count,
    total_freight,
    total_weight_kg,
    freight_per_kg,
    mean_item_freight_per_kg,
    mean_freight_to_price,
    mean_distance_km,
    quarter_baseline_fpk,
    relative_to_quarter,

    RANK() OVER (
        PARTITION BY purchase_quarter
        ORDER BY freight_per_kg DESC
    ) AS rank_in_quarter,

    COUNT(*) OVER (
        PARTITION BY purchase_quarter
    ) AS lanes_in_quarter,

    LAG(purchase_quarter) OVER (
        PARTITION BY lane
        ORDER BY purchase_quarter
    ) AS prev_quarter,

    LAG(freight_per_kg) OVER (
        PARTITION BY lane
        ORDER BY purchase_quarter
    ) AS prev_freight_per_kg,

    LAG(relative_to_quarter) OVER (
        PARTITION BY lane
        ORDER BY purchase_quarter
    ) AS prev_relative,

    ROUND(
        100 * (
            freight_per_kg -
            LAG(freight_per_kg) OVER (
                PARTITION BY lane
                ORDER BY purchase_quarter
            )
        )
        /
        LAG(freight_per_kg) OVER (
            PARTITION BY lane
            ORDER BY purchase_quarter
        ),
        1
    ) AS change_pct,

    relative_to_quarter -
    LAG(relative_to_quarter) OVER (
        PARTITION BY lane
        ORDER BY purchase_quarter
    ) AS relative_change

FROM with_baseline
"""

lane_quarter_sql = spark.sql(sql_query)

print("Qualifying lane-quarters (SQL):", lane_quarter_sql.count())

Qualifying lane-quarters (SQL): 348


In [42]:
# Save the SQL result as a temporary view so it is easier to query
lane_quarter_sql.createOrReplaceTempView("lane_quarter_sql")


print("=== 2018-Q2: lanes ranked by freight charged per kilogram (SQL) ===")

spark.sql("""
    SELECT
        rank_in_quarter,
        lane,
        item_count,
        ROUND(freight_per_kg, 2) AS freight_per_kg,
        ROUND(mean_item_freight_per_kg, 2) AS mean_item_ratio,
        ROUND(relative_to_quarter, 2) AS relative_to_quarter,
        ROUND(mean_distance_km, 0) AS avg_dist_km
    FROM lane_quarter_sql
    WHERE purchase_quarter = '2018-Q2'
    ORDER BY rank_in_quarter
    LIMIT 15
""").show(truncate=False)

=== 2018-Q2: lanes ranked by freight charged per kilogram (SQL) ===
+---------------+--------+----------+--------------+---------------+-------------------+-----------+
|rank_in_quarter|lane    |item_count|freight_per_kg|mean_item_ratio|relative_to_quarter|avg_dist_km|
+---------------+--------+----------+--------------+---------------+-------------------+-----------+
|1              |MA -> SP|53        |61.07         |67.54          |6.16               |2329.0     |
|2              |PE -> MG|30        |49.21         |55.39          |4.96               |1700.0     |
|3              |PE -> SP|35        |41.06         |50.92          |4.14               |2118.0     |
|4              |RJ -> PE|30        |36.17         |62.69          |3.65               |1849.0     |
|5              |SP -> RN|70        |33.34         |116.88         |3.36               |2271.0     |
|6              |SP -> MA|85        |24.4          |78.59          |2.46               |2208.0     |
|7              |RJ -> 

In [43]:
print("=== Largest movements relative to the quarterly platform baseline (SQL) ===")

spark.sql("""
    SELECT
        lane,
        prev_quarter,
        purchase_quarter,
        item_count,
        ROUND(prev_relative, 2) AS prev_relative,
        ROUND(relative_to_quarter, 2) AS relative_to_quarter,
        ROUND(relative_change, 2) AS relative_change,
        change_pct,
        rank_in_quarter,
        lanes_in_quarter
    FROM lane_quarter_sql
    WHERE prev_relative IS NOT NULL
    ORDER BY ABS(relative_change) DESC
    LIMIT 15
""").show(truncate=False)

=== Largest movements relative to the quarterly platform baseline (SQL) ===
+--------+------------+----------------+----------+-------------+-------------------+---------------+----------+---------------+----------------+
|lane    |prev_quarter|purchase_quarter|item_count|prev_relative|relative_to_quarter|relative_change|change_pct|rank_in_quarter|lanes_in_quarter|
+--------+------------+----------------+----------+-------------+-------------------+---------------+----------+---------------+----------------+
|SP -> TO|2017-Q2     |2017-Q4         |32        |1.19         |3.53               |2.34           |222.4     |1              |58              |
|SP -> TO|2017-Q4     |2018-Q1         |48        |3.53         |1.41               |-2.12          |-59.7     |19             |59              |
|SP -> RN|2018-Q1     |2018-Q2         |70        |1.87         |3.36               |1.5            |90.2      |5              |60              |
|SP -> MA|2017-Q2     |2017-Q3         |98      


### Notes on the SQL implementation

We can see immediately that there are a few key differences in the SQL version when comparing it to the DataFrame approach.

**The CTEs follow the same steps as the analysis.** Each `WITH` block handles one distinct part of the process: 
- Filtering to usable rows
- Calculating the quarterly baseline
- Aggregating to lane-quarter level
- Applying the minimum volume rule
- Joining the baseline back on

The final `SELECT` then adds the ranking and previous-period comparisons, making the SQL version easier to compare directly with the DataFrame implementation.

There are 3 key aspects in our comparison of the DataFrame and SQL versions that we should highlight:

**1. The post-aggregation filter:** In the DataFrame version, the 30-item minimum is applied using `.filter()` after `.agg()`. However, in SQL, we are able to implement the same rule using `HAVING`, which makes it immediately clear to a reader that the condition applies to the grouped result rather than to individual rows.

**2. Repeated window expression:** In the final `SELECT` call, `LAG(freight_per_kg)` is used both as an output column and then again when calculating `change_pct`. This is due to the fact that the alias created in the same `SELECT` cannot be reused immediately, and so the window expression appears more than once. In this regard, the DataFrame version is a little easier to read.

**3. Minimum threshold definition:** The DataFrame version uses the `MIN_ITEMS` variable, while the SQL query contains the value `30` in the `HAVING` clause. This works fine for our current analysis, but changing the threshold would require editing the SQL text unless the query was built using a variable.



---

### Comparing the DataFrame API and Spark SQL

Now that we have implemented the same analysis using both approaches, our next step is to consider a side by side comparison of how they align, and also how they differ in practice.

The fundamental difference we need to consider here is not what they are capable of doing, since it is clear that both the DataFrame API and Spark SQL are capable of the analysis (and also, they are processed by the same Catalyst optimiser). Instead, we are concerned with the key differences in how the analysis is written, followed and changed.

#### **Where the DataFrame API works well**

Looking first at the DataFrame version, it is important to note the relative ease at which intermediate columns could be created and reused at later points. For example, `prev_freight_per_kg` was created once and then we could use it again later for `change_pct`. On the other hand, in our SQL query, the `LAG()` expression needed to be repeated in the final `SELECT`, which makes that section longer and means the same logic needs to be redefined in several places.

Parameters are also easier to keep separate from the analysis under the DataFrame version. We were able to define `MIN_ITEMS = 30` once and then leverage that defined reference, which is important since by having a named variable, it makes it easier to later test different thresholds.

The DataFrame approach was also useful while building our analysis. We were able to separtely inspect objects such as `base_weighted`, `lane_quarter_sizes` and `lane_quarter` , which made it straightforward to check intermediate results and decide on the minimum group size.

#### **Where Spark SQL works well**

While the DataFrame version certainly offers key advantages, it important to also note that the SQL version gives a clearer view of the finished analysis as a whole, which can be a significant benefit when conducting complex queries. 

In particular, the CTEs divide the query into named stages — `weighted_items`, `quarter_baseline`, `lane_quarter` and `with_baseline` — before the window calculations are applied. This makes it possible to follow most of the final analytical pipeline from top to bottom in one place.

Furthermore, the distinction between row-level and group-level filtering is also clearer. SQL uses `WHERE` for filtering individual rows and `HAVING` for filtering after aggregation. In the DataFrame version both are written using `.filter()`, with their position in the transformation sequence determining when they are applied.

SQL also utilises codig syntax and language that is arguably a much more intuitive to learn, making it a very accessible option for newer developers.

#### **Which approach suited this analysis?**

For our specific use case, the DataFrame API was more useful while we were developing and testing the analysis. We were able to build the pipeline gradually, inspect intermediate results and adjust individual steps without rewriting the whole query.

Once our logic had been settled, the SQL version provided a more compact view of the finished process because the CTEs keep each stage together in a single query.

Given this, it arguably makes little sense for us to choose one approach purely on the basis of performance. This is due to that fact that when considering the question of equivalency, both use Spark's underlying execution engine, so readability and the type of work being done are more useful reasons for choosing between them.

However, one downside of maintaining both versions is that any future change to the analysis needs to be made twice. Therefore, the equivalence check in Part A.4 of our analysis is critical when testing because it confirms that the two implementations currently produce the same result.

 
 ---

 ## Part A.4: Results validation

 Now that we have implemented our analysis using both the DataFrame API and Spark SQL, we need to check that they actually produce the same result. 
 
 Since differences could exist anywhere in the outputs or could even be hidden by rounding differences, simply looking at the summary statistics, or even just the top few rows is not sufficient. We therefore want to compare the full results rather than relying on a simple visual check. 
 
 As such, our validation aims to look at four things: 

 1. Both outputs have the same number of rows, the same lane-quarter keys and matching column types. 
 2. Rows from one result can be matched to the other using the full, unrounded values. 
 3. Numeric columns are comparable, with some small allowances made for a small floating-point difference in case tiny calculation differences appear. 
 4. After applying the same ordering, the two complete outputs match. 
 
 Before running these checks, we need to first ensure that both results are reduced to the same set of columns in the same order so that we are comparing like with like.

In [44]:
# Use the same columns from both results
compare_cols = [
    "lane",
    "purchase_quarter",
    "item_count",
    "order_count",
    "total_freight",
    "total_weight_kg",
    "freight_per_kg",
    "mean_item_freight_per_kg",
    "quarter_baseline_fpk",
    "relative_to_quarter",
    "rank_in_quarter",
    "lanes_in_quarter",
    "prev_quarter",
    "prev_freight_per_kg",
    "prev_relative",
    "change_pct",
    "relative_change"
]

df_result = lane_quarter_ranked.select(compare_cols)
sql_result = lane_quarter_sql.select(compare_cols)

In [45]:
# Check the basic shape of both outputs
df_rows = df_result.count()
sql_rows = sql_result.count()

print("DataFrame rows:", df_rows)
print("SQL rows:", sql_rows)
print("Column types match:", df_result.dtypes == sql_result.dtypes)


# Check that both contain the same lane-quarter keys
df_keys = df_result.select(
    "lane",
    "purchase_quarter"
).distinct()

sql_keys = sql_result.select(
    "lane",
    "purchase_quarter"
).distinct()

print("DataFrame keys:", df_keys.count())
print("SQL keys:", sql_keys.count())

print(
    "Keys only in DataFrame:",
    df_keys.exceptAll(sql_keys).count()
)

print(
    "Keys only in SQL:",
    sql_keys.exceptAll(df_keys).count()
)

DataFrame rows: 348
SQL rows: 348
Column types match: True
DataFrame keys: 348
SQL keys: 348
Keys only in DataFrame: 0
Keys only in SQL: 0


In [46]:
# Check whether any complete rows differ
df_only = df_result.exceptAll(sql_result)
sql_only = sql_result.exceptAll(df_result)

print(
    "Rows in DataFrame result but not SQL:",
    df_only.count()
)

print(
    "Rows in SQL result but not DataFrame:",
    sql_only.count()
)

Rows in DataFrame result but not SQL: 0
Rows in SQL result but not DataFrame: 0


In [49]:
# Match rows by lane and quarter so the numeric values can be compared
matched = df_result.alias("d").join(
    sql_result.alias("s"),
    on=["lane", "purchase_quarter"],
    how="inner"
)

# Compare the numeric columns for each matched lane-quarter
diffs = matched.select(
    F.max(
        F.abs(F.col("d.total_freight") - F.col("s.total_freight"))
    ).alias("total_freight"),

    F.max(
        F.abs(F.col("d.total_weight_kg") - F.col("s.total_weight_kg"))
    ).alias("total_weight_kg"),

    F.max(
        F.abs(F.col("d.freight_per_kg") - F.col("s.freight_per_kg"))
    ).alias("freight_per_kg"),

    F.max(
        F.abs(
            F.col("d.mean_item_freight_per_kg")
            - F.col("s.mean_item_freight_per_kg")
        )
    ).alias("mean_item_freight_per_kg"),

    F.max(
        F.abs(
            F.col("d.quarter_baseline_fpk")
            - F.col("s.quarter_baseline_fpk")
        )
    ).alias("quarter_baseline_fpk"),

    F.max(
        F.abs(
            F.col("d.relative_to_quarter")
            - F.col("s.relative_to_quarter")
        )
    ).alias("relative_to_quarter"),

    F.max(
        F.abs(
            F.col("d.prev_freight_per_kg")
            - F.col("s.prev_freight_per_kg")
        )
    ).alias("prev_freight_per_kg"),

    F.max(
        F.abs(
            F.col("d.prev_relative")
            - F.col("s.prev_relative")
        )
    ).alias("prev_relative"),

    F.max(
        F.abs(F.col("d.change_pct") - F.col("s.change_pct"))
    ).alias("change_pct"),

    F.max(
        F.abs(
            F.col("d.relative_change")
            - F.col("s.relative_change")
        )
    ).alias("relative_change")
)

diffs.show(vertical=True, truncate=False)

-RECORD 0------------------------------------------
 total_freight            | 2.764863893389702E-10  
 total_weight_kg          | 3.2741809263825417E-11 
 freight_per_kg           | 3.108624468950438E-14  
 mean_item_freight_per_kg | 5.6843418860808015E-14 
 quarter_baseline_fpk     | 0.0                    
 relative_to_quarter      | 3.3306690738754696E-15 
 prev_freight_per_kg      | 3.108624468950438E-14  
 prev_relative            | 3.3306690738754696E-15 
 change_pct               | 0.0                    
 relative_change          | 3.4416913763379853E-15 



In [50]:
# Find the largest difference across all numeric columns
diff_values = diffs.collect()[0]

max_diff = max(
    value for value in diff_values
    if value is not None
)

print("Largest difference in any numeric column:", max_diff)
print("Within tolerance of 1e-9:", max_diff < 1e-9)

Largest difference in any numeric column: 2.764863893389702e-10
Within tolerance of 1e-9: True


In [48]:
# Final check after putting both results in the same order
df_ordered = df_result.orderBy(
    "purchase_quarter",
    "lane"
)

sql_ordered = sql_result.orderBy(
    "purchase_quarter",
    "lane"
)

same_results = df_ordered.collect() == sql_ordered.collect()

print("Ordered results identical:", same_results)

Ordered results identical: True
